In [1]:
%pip install -q langchain-openai python-dotenv langgraph langchain langchain-classic langchain-community pypdf docx2txt py-zerox


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q langchain-text-splitters pymupdf


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader  
import os, pprint, requests

In [5]:
pdf_file_path = './meeting.pdf'
loader = PyPDFLoader(pdf_file_path)
pages = []
async for page in loader.alazy_load():
    pages.append(page)

In [6]:
pages

[Document(metadata={'producer': 'Skia/PDF m137', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/137.0.0.0 Safari/537.36', 'creationdate': '2025-10-27T13:27:42+00:00', 'title': '회고', 'moddate': '2025-10-27T13:27:42+00:00', 'source': './meeting.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='⏪\n회고\n태그 회고\n날짜\n좋았던  점 (Good)\n용범\n우선 , 정말  뜻  깊은  한  주를  보낸  것  같아서  너무  뿌듯한  마음입니다 . 간식을  모두와  함께  먹으며 , 당이  떨어지지  않는  환경  속에서  개발을 \n할  수  있어서  좋았습니다 . 모두  건강했으면  좋겠습니다 . \n김나영  교육생의  발표  실력에  감탄을  금치  못했습니다 . \n연지윤  교육생의  PPT 구성  실력에  감탄을  금치  못했습니다 . \n배울  점이  많습니다 . \n예림  \n요즘  지각  안하려고  20 분  일찍  오고  있습니다 !\nAI 는  아니지만  공부하면서  LLM, 랭체인 , 라그 , 각종  알고리즘  등  알게  되어서  좋았습니다 . 요즘  공고에  AI 해본  백엔드  우대사항이  정말 \n많더라고요  …\n간식이  많아져서  행복합니다 . 용범님의  애정 ? 감사합니다 .\n같이  감자빵  사러  가는  거  약간  힐링입니다 .\n나영님의  발표가  인상  깊었습니다 .\n지윤님과  주혜님의  주도하에  PPT 가  예쁘게  만들어졌습니다 .\n이번  주도  마찬가지로  주혜님이  귀엽습니다 .\n코어  시간은  스스로  잘  지키고  있는  것  같습니다 !\n의균  \n

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyMuPDFLoader

# PDF 불러오기
pdf_path = "./meeting.pdf"
loader = PyMuPDFLoader(pdf_path)
pages = loader.load()

# 모든 페이지를 하나의 텍스트로 합치기
pdf_text = "\n\n".join([page.page_content for page in pages])

print(f"총 {len(pages)} 페이지, 텍스트 길이: {len(pdf_text)}자")



총 4 페이지, 텍스트 길이: 2969자


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
chunks = splitter.split_text(pdf_text)
print("총 chunk 수:", len(chunks))

총 chunk 수: 2


In [9]:
from langchain_core.prompts import PromptTemplate

with open("prompt_docCo.txt", "r", encoding="utf-8") as f:
    prompt_text = f.read()


prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_text
)

question = "이 문서에서 핵심적으로 주장하는 내용은 무엇인가요?"
formatted_prompt = prompt_template.format(context=pdf_text[:8000], question=question)

In [ ]:
from datetime import datetime
from pathlib import Path

MODEL_NAME = "gpt-5"
OUTPUT_DIR = "./outputs"
PDF_PATH = "./meeting.pdf"

# 5) LLM 호출
llm = init_chat_model(MODEL_NAME, model_provider="openai")
response = llm.invoke(formatted_prompt).content

# 6) Markdown 저장
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

def safe_filename(s: str) -> str:
    bad = r'\/:*?"<>|'
    for ch in bad:
        s = s.replace(ch, "_")
    return s.strip() or "output"

base_title = Path(PDF_PATH).stem  
timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
md_filename = f"{safe_filename(base_title)}_{timestamp}.md"
md_path = Path(OUTPUT_DIR) / md_filename

# (선택) frontmatter/메타 정보 추가
front_matter = f"""---
title: "{base_title} 요약 리포트"
source_pdf: "{PDF_PATH}"
generated_at: "{datetime.now().isoformat(timespec='seconds')}"
model: "{MODEL_NAME}"
---
"""

In [11]:
with open(md_path, "w", encoding="utf-8") as f:
    f.write(front_matter)
    f.write("\n")
    f.write(response)

print(f"✅ Markdown 저장 완료: {md_path}")

✅ Markdown 저장 완료: outputs/meeting_2025-10-28_0025.md
